# Task 3: Notebook to demonstrate Zero shot and Few shot Learning

In [1]:
from dotenv import load_dotenv
import pandas as pd
import os
from langchain_groq.chat_models import ChatGroq
import time
import re
import numpy as np

load_dotenv()

Groq_Token = os.getenv('GROQ_TOKEN')  # Do not share this key with anyone
groq_models = {"llama3-70b": "llama3-70b-8192", "mixtral": "mixtral-8x7b-32768", "gemma-7b": "gemma-7b-it",
               "llama3.1-70b": "llama-3.1-70b-versatile", "llama3-8b": "llama3-8b-8192",
               "llama3.1-8b": "llama-3.1-8b-instant", "gemma-9b": "gemma2-9b-it"}

**NOTE : DO NOT SHARE THE API KEY WITH ANYONE. DO NOT COMMIT THE API KEY TO GITHUB.**

Always do a sanity check before committing the code to github. If the key is found in the code, you will be penalized with a 0.5 marks deduction.

Q1) Demonstrate how to use Zero-Shot Learning and Few-Shot Learning to classify human activities based on the featurized accelerometer data. Qualitatively demonstrate the performance of Few-Shot Learning with Zero-Shot Learning. Which method performs better? Why? [1 marks]

### Zero Shot 

In [2]:
import os

correct_prediction = 0
total = 0

test_dir = './Combined/Test/'
activities = ['LAYING', 'SITTING', 'STANDING', 'WALKING', 'WALKING_DOWNSTAIRS', 'WALKING_UPSTAIRS']

print("Initiating Zero Shot Learning on UCI HAR Dataset")
for x in activities:
    for y in os.listdir(test_dir + x):
        data = pd.read_csv(test_dir + x + '/' + y)
        query = f"""* You are a physical activity classification model. 
* Your task is to analyze the data given in the csv file. The data is from an accelerometer where
first 10 seconds of activity, translating to the initial 500 data samples due to a sampling rate of 50Hz is given.
Each datapoint has acceleration values in the x, y, and z directions.
classify the activity as 'LAYING', 'SITTING', 'STANDING', 'WALKING', 'WALKING_DOWNSTAIRS', 'WALKING_UPSTAIRS'. 
* Provide the activity label only as a single word.

Sentence: {data}"""

        model_name = "llama3.1-70b"
        llm = ChatGroq(model=groq_models[model_name], api_key=Groq_Token, temperature=0)
        answer = llm.invoke(query)

        if answer.content == x:
            correct_prediction += 1
        total += 1

print("Accuracy of Zero Shot Learning on UCI HAR Dataset: ", correct_prediction / total)

Initiating Zero Shot Learning on UCI HAR Dataset
Accuracy of Zero Shot Learning on UCI HAR Dataset:  0.18518518518518517


### Few Shot

In [2]:
# Load your data
X_train = pd.read_csv('./UCI HAR Dataset/train/X_train.txt', delim_whitespace=True, header=None)
Y_train = pd.read_csv('./UCI HAR Dataset/train/y_train.txt', header=None)
X_test = pd.read_csv('./UCI HAR Dataset/test/X_test.txt', delim_whitespace=True, header=None)
Y_test = pd.read_csv('./UCI HAR Dataset/test/y_test.txt', header=None)

correct_prediction = 0
total = 0

examples = []

# Here we are selecting 6 examples from the training data to be used for few shot learning
# so that each activity is there in the examples

for i in [(10, 5), (32, 4), (56, 6), (85, 1), (134, 3), (162, 2)]:
    examples.append((X_train.iloc[i[0]], Y_train.iloc[i[1]]))

print("Initiating Few Shot Learning on UCI HAR Dataset")

X_test = X_test.sample(frac=1).reset_index(drop=True)
for index, row in X_test.iterrows():

    query = """You are a physical activity classification model. 
Your task is to analyze the featurized data given in the csv files X_train and Y_train.
Learn how the mapping is from X_train to Y_train and classify the X_test data. 
Output should be just a number which lies between 1 to 6 corresponding to the activity. 
Output only a number as a single word nothing else, no explanation, no code. """

    for i in range(len(examples)):
        query += f"Example {i + 1}:\nX Train data = {examples[i][0]}\nY Train data = {examples[i][1]}\n"

    query += f"\nGiven below is the x test data, predict the activity label as a number between 1 to 6:\nX Test data = {row}\n"

    model_name = "llama3.1-70b"
    llm = ChatGroq(model=groq_models[model_name], api_key=Groq_Token, temperature=0)

    try:
        answer = llm.invoke(query)
        if int(re.search(r'\d+', answer.content.strip()).group()) == Y_test.iloc[index][0]:
            correct_prediction += 1
        total += 1
    
        if total == 30:
            print(f"Accuracy after {total} samples: {correct_prediction / total}")
            break
    
    except Exception as e:
        print(e)
        time.sleep(100)

print(f"Final Accuracy: {correct_prediction / total}")

/tmp/ipykernel_158557/427823508.py:2: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  X_train = pd.read_csv('./UCI HAR Dataset/train/X_train.txt', delim_whitespace=True, header=None)
/tmp/ipykernel_158557/427823508.py:4: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  X_test = pd.read_csv('./UCI HAR Dataset/test/X_test.txt', delim_whitespace=True, header=None)


Initiating Few Shot Learning on UCI HAR Dataset
Accuracy after 30 samples: 0.5333333333333333
Final Accuracy: 0.5333333333333333


Q2) Quantitatively compare the accuracy of Few-Shot Learning with Decision Trees (You may use a subset of the test set if you encounter rate-limiting issues). Which method performs better? Why? [1 marks]

### Decision Tree

In [3]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

X_train = pd.read_csv('./UCI HAR Dataset/train/X_train.txt', delim_whitespace=True, header=None)
Y_train = pd.read_csv('./UCI HAR Dataset/train/y_train.txt', header=None)
X_test = pd.read_csv('./UCI HAR Dataset/test/X_test.txt', delim_whitespace=True, header=None)
Y_test = pd.read_csv('./UCI HAR Dataset/test/y_test.txt', header=None)

/tmp/ipykernel_158557/4261227942.py:5: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  X_train = pd.read_csv('./UCI HAR Dataset/train/X_train.txt', delim_whitespace=True, header=None)
/tmp/ipykernel_158557/4261227942.py:7: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  X_test = pd.read_csv('./UCI HAR Dataset/test/X_test.txt', delim_whitespace=True, header=None)


In [4]:
clf = DecisionTreeClassifier(random_state=42)
clf.fit(X_train, Y_train)
y_pred = clf.predict(X_test[:30])
print("Accuracy for our data: ", accuracy_score(y_pred, Y_test[:30]))

Accuracy for our data:  0.9


Q3) What are the limitations of Zero-Shot Learning and Few-Shot Learning in the context of classifying human activities based on featurized accelerometer data? [1 marks]


The mappings from features to the activity classes is abiguous and complex, so learning a mapping from the input data to output data
using Zero Shot Learning and Few Shot Learning is hard. Because both of these methods assume that there is a well defined mapping
that can be found between the input(features) and the output(activity classes). And for Human Activity Recognition that is not the case.

Another thing to notice is that even a small change in accelerometer data may lead to the model classifying into a completely different activity.
This is because Human Activities can have similar accelerometer data for activities like standing, sitting, etc.

For Few Shot Learning we observe that the model can learn bias towards the activity classes that it has already seen. So instead of actually learning a mapping from input to output. It just gives the output based on its biasness towards the seen activities. This can lead to misclassifications, particularly when the unseen class is not sufficiently distinct in the semantic space.

Q4) What does the model classify when given input from an entirely new activity that it hasn't seen before? [0.5 mark]


In [5]:
correct_prediction = 0

examples = []

# Here we are selecting 5 examples from the training data to be used for few shot learning
# so that 5 of the activities are these as examples
# The example for the 6th activity will be the new activity that the model has never seen before

for i in [(10, 5), (32, 4), (56, 6), (85, 1), (134, 3), (162, 2)]:
    examples.append((X_train.iloc[i[0]], Y_train.iloc[i[1]]))

query = f"""
* You are a physical activity classification model. 
* Your task is to analyze the data given in the csv file. The data is from an accelerometoe where
  first 10 seconds of activity are recorded and we have featurized the data into 561 features.
  classify the activity as 'LAYING', 'SITTING', 'STANDING', 'WALKING', 'WALKING_DOWNSTAIRS', 'WALKING_UPSTAIRS'. 
* Provide the activity label only as a single word.
"""

for i in range(len(examples) - 1):
    query += f"Example {i + 1}:\nX Train data = {examples[i][0]}\nY Train data = {examples[i][1]}\n"

query += f"\nGiven below is the x test data, predict the activity label as a number between 1 to 6:\nX Test data = {examples[5][0]}\n"

model_name = "llama3-70b"
llm = ChatGroq(model=groq_models[model_name], api_key=Groq_Token, temperature=0)
answer = llm.invoke(query)
print(answer.content)

Based on the provided X Test data, I predict the activity label as:

WALKING


Q5) Test the model with random data (ensuring the data has the same dimensions and range as the previous input) and report the results. [0.5 mark]

In [6]:
X_min, X_max = X_train.min(), X_train.max()
Y_min, Y_max = Y_train.min(), Y_train.max()

X_test_random = np.random.uniform(low=X_min, high=X_max, size=(7352, 561))
Y_test_random = np.random.uniform(low=Y_min, high=Y_max, size=(7352, 1))

In [7]:
correct_prediction = 0
total = 0

query = f"""
* You are a physical activity classification model. 
* Your task is to analyze the data given in the csv file. The data is from an accelerometer where
  first 10 seconds of activity are recorded and we have featurized the data into 561 features.
  classify the activity as 'LAYING', 'SITTING', 'STANDING', 'WALKING', 'WALKING_DOWNSTAIRS', 'WALKING_UPSTAIRS'. 


Sentence: {X_test_random[0]}
"""

model_name = "llama3-70b"
llm = ChatGroq(model=groq_models[model_name], api_key=Groq_Token, temperature=0)
answer = llm.invoke(query)
print(answer.content)

I'm a physical activity classification model, and I'll analyze the given data to classify the activity into one of the six categories: 'LAYING', 'SITTING', 'STANDING', 'WALKING', 'WALKING_DOWNSTAIRS', or 'WALKING_UPSTAIRS'.

After analyzing the data, I'm going to classify the activity as **WALKING**.

Please note that this classification is based on the patterns and features extracted from the given data, and the accuracy of the classification may vary depending on the quality and representativeness of the data.
